# PRDC Structuring: Power Reverse Dual Currency Notes

A PRDC pays a coupon linked to FX: `max(fixed + participation × (FX/strike - 1), 0)`.

Popular with Japanese investors: receive high JPY coupons when USD/JPY is high. The issuer is short a strip of FX options — and usually embeds a call right to limit exposure.

**What we build:**
1. Non-callable PRDC pricing (3-factor MC: domestic rate + foreign rate + FX)
2. Callable PRDC (LSM for issuer call)
3. Correlation sensitivity — how rho_FX_dom changes the price
4. Call value — what does the embedded call option cost?
5. Greek profiles — FX delta, IR delta

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), "python"))

import numpy as np
from pricebook.prdc import prdc_price, callable_prdc
from pricebook.viz import configure_theme
from pricebook.viz._backend import apply_theme, create_figure

configure_theme(seaborn_style="whitegrid", seaborn_context="notebook")

# Base parameters: JPY investor, USD/JPY FX
SPOT = 155.0        # USD/JPY spot (Jul 2024)
R_DOM = 0.001       # JPY rate (near zero)
R_FOR = 0.053       # USD rate (SOFR)
VOL_FX = 0.10       # USD/JPY implied vol
VOL_DOM = 0.005     # JPY rate vol
VOL_FOR = 0.012     # USD rate vol
RHO_FX_DOM = 0.15   # FX-JPY rate correlation
RHO_FX_FOR = 0.25   # FX-USD rate correlation
RHO_DOM_FOR = 0.30  # JPY-USD rate correlation

# Note structure
NOTIONAL = 100.0
FIXED = 0.06        # 6% fixed component
PARTICIPATION = 1.0 # 100% FX participation
FX_STRIKE = 155.0   # ATM strike
T = 10              # 10-year maturity
N_COUPONS = 10      # annual coupons
N_PATHS = 10_000
N_STEPS = 120

print(f"PRDC Note: {T}Y, {FIXED*100:.0f}% + {PARTICIPATION*100:.0f}% × (USDJPY/{FX_STRIKE:.0f} - 1)")
print(f"Spot USDJPY: {SPOT}, JPY rate: {R_DOM*100:.1f}%, USD rate: {R_FOR*100:.1f}%")
print(f"FX vol: {VOL_FX*100:.0f}%, Paths: {N_PATHS:,}")

## 1. Non-callable PRDC pricing

In [ ]:
result = prdc_price(
    SPOT, R_DOM, R_FOR, VOL_FX, VOL_DOM, VOL_FOR,
    RHO_FX_DOM, RHO_FX_FOR, RHO_DOM_FOR,
    NOTIONAL, FIXED, PARTICIPATION, FX_STRIKE,
    T, N_COUPONS, N_PATHS, N_STEPS,
)

print(f"Non-callable PRDC Price: {result.price:.2f}")
print(f"Mean coupon:             {result.mean_coupon*100:.2f}%")
print(f"FX delta (per 1 JPY):    {result.fx_delta:.4f}")
print(f"IR delta (per 1bp):      {result.ir_delta:.4f}")
print(f"Premium over par:        {result.price - NOTIONAL:+.2f}")

## 2. Callable PRDC — issuer call via LSM

In [ ]:
callable_result = callable_prdc(
    SPOT, R_DOM, R_FOR, VOL_FX, VOL_DOM, VOL_FOR,
    RHO_FX_DOM, RHO_FX_FOR, RHO_DOM_FOR,
    NOTIONAL, FIXED, PARTICIPATION, FX_STRIKE,
    T, N_COUPONS,
    call_dates=list(range(2, N_COUPONS + 1)),  # callable from year 2
    n_paths=N_PATHS, n_steps=N_STEPS,
)

call_value = callable_result.price_no_call - callable_result.price

print(f"Callable PRDC Price:     {callable_result.price:.2f}")
print(f"Non-callable price:      {callable_result.price_no_call:.2f}")
print(f"Call option value:       {call_value:.2f} ({call_value/NOTIONAL*100:.1f}% of notional)")
print(f"Call probability:        {callable_result.call_probability*100:.1f}%")
print(f"Mean call time:          {callable_result.mean_call_time:.1f}Y")
print(f"\nThe investor gives up {call_value:.2f} by accepting the callable feature.")
print(f"In return, the structurer can offer a higher coupon or lower strike.")

## 3. Correlation sensitivity

The PRDC price is highly sensitive to ρ(FX, domestic rate). When FX and JPY rates are positively correlated, a JPY weakening (good for the coupon) coincides with higher JPY rates (lower PV) — natural hedge.

In [ ]:
rho_range = [-0.4, -0.2, 0.0, 0.15, 0.3, 0.5]
prices_by_rho = []
call_prices_by_rho = []

print(f"{'rho_FX_dom':>10}  {'Non-call':>9}  {'Callable':>9}  {'Call Val':>9}  {'Call Prob':>9}")
print(f"{'─'*10}  {'─'*9}  {'─'*9}  {'─'*9}  {'─'*9}")

for rho in rho_range:
    r = prdc_price(
        SPOT, R_DOM, R_FOR, VOL_FX, VOL_DOM, VOL_FOR,
        rho, RHO_FX_FOR, RHO_DOM_FOR,
        NOTIONAL, FIXED, PARTICIPATION, FX_STRIKE,
        T, N_COUPONS, N_PATHS, N_STEPS,
    )
    cr = callable_prdc(
        SPOT, R_DOM, R_FOR, VOL_FX, VOL_DOM, VOL_FOR,
        rho, RHO_FX_FOR, RHO_DOM_FOR,
        NOTIONAL, FIXED, PARTICIPATION, FX_STRIKE,
        T, N_COUPONS, n_paths=N_PATHS, n_steps=N_STEPS,
    )
    cv = cr.price_no_call - cr.price
    prices_by_rho.append(r.price)
    call_prices_by_rho.append(cr.price)
    print(f"{rho:>+10.2f}  {r.price:>9.2f}  {cr.price:>9.2f}  {cv:>9.2f}  {cr.call_probability*100:>8.1f}%")

# Plot
with apply_theme():
    fig, ax = create_figure(1)
    ax.plot(rho_range, prices_by_rho, 'o-', linewidth=2, label='Non-callable')
    ax.plot(rho_range, call_prices_by_rho, 's--', linewidth=2, label='Callable')
    ax.axhline(NOTIONAL, color='black', linewidth=0.5, linestyle=':')
    ax.set_xlabel('ρ(FX, domestic rate)')
    ax.set_ylabel('PRDC Price')
    ax.set_title('PRDC Price Sensitivity to FX-Rate Correlation')
    ax.legend(fontsize=9)
    fig.tight_layout()

## 4. FX spot sensitivity — the delta profile

In [ ]:
spots = np.linspace(120, 190, 15)
prices_by_spot = []
call_prices_by_spot = []

print(f"{'USDJPY':>8}  {'Non-call':>9}  {'Callable':>9}  {'Δ Non-call':>11}")
print(f"{'─'*8}  {'─'*9}  {'─'*9}  {'─'*11}")

prev = None
for s in spots:
    r = prdc_price(
        s, R_DOM, R_FOR, VOL_FX, VOL_DOM, VOL_FOR,
        RHO_FX_DOM, RHO_FX_FOR, RHO_DOM_FOR,
        NOTIONAL, FIXED, PARTICIPATION, FX_STRIKE,
        T, N_COUPONS, N_PATHS, N_STEPS,
    )
    cr = callable_prdc(
        s, R_DOM, R_FOR, VOL_FX, VOL_DOM, VOL_FOR,
        RHO_FX_DOM, RHO_FX_FOR, RHO_DOM_FOR,
        NOTIONAL, FIXED, PARTICIPATION, FX_STRIKE,
        T, N_COUPONS, n_paths=N_PATHS, n_steps=N_STEPS,
    )
    delta_str = f"{(r.price - prev) / (spots[1] - spots[0]):+.3f}" if prev is not None else "—"
    prices_by_spot.append(r.price)
    call_prices_by_spot.append(cr.price)
    prev = r.price
    print(f"{s:>8.0f}  {r.price:>9.2f}  {cr.price:>9.2f}  {delta_str:>11}")

# Plot
with apply_theme():
    fig, ax = create_figure(1)
    ax.plot(spots, prices_by_spot, 'o-', linewidth=2, label='Non-callable')
    ax.plot(spots, call_prices_by_spot, 's--', linewidth=2, label='Callable')
    ax.axhline(NOTIONAL, color='black', linewidth=0.5, linestyle=':')
    ax.axvline(SPOT, color='tab:gray', linewidth=0.5, linestyle=':', label=f'Spot ({SPOT})')
    ax.set_xlabel('USDJPY Spot')
    ax.set_ylabel('PRDC Price')
    ax.set_title('PRDC Price vs FX Spot')
    ax.legend(fontsize=9)
    fig.tight_layout()

## 5. Structuring: what coupon can you offer?

The structurer sets the fixed coupon so that the non-callable PRDC prices at par (100). A higher participation or lower strike allows a higher fixed coupon.

In [ ]:
# Vary the fixed coupon to find par pricing
test_coupons = np.arange(0.02, 0.10, 0.005)
prices_by_cpn = []

for cpn in test_coupons:
    r = prdc_price(
        SPOT, R_DOM, R_FOR, VOL_FX, VOL_DOM, VOL_FOR,
        RHO_FX_DOM, RHO_FX_FOR, RHO_DOM_FOR,
        NOTIONAL, cpn, PARTICIPATION, FX_STRIKE,
        T, N_COUPONS, N_PATHS, N_STEPS,
    )
    prices_by_cpn.append(r.price)

# Interpolate for par coupon
prices_arr = np.array(prices_by_cpn)
par_idx = np.argmin(np.abs(prices_arr - NOTIONAL))
if par_idx > 0 and par_idx < len(test_coupons) - 1:
    # Linear interpolation
    if prices_arr[par_idx] > NOTIONAL:
        lo, hi = par_idx - 1, par_idx
    else:
        lo, hi = par_idx, par_idx + 1
    frac = (NOTIONAL - prices_arr[lo]) / (prices_arr[hi] - prices_arr[lo])
    par_coupon = test_coupons[lo] + frac * (test_coupons[hi] - test_coupons[lo])
else:
    par_coupon = test_coupons[par_idx]

print(f"Par coupon (non-callable): {par_coupon*100:.2f}%")
print(f"This is the fixed coupon that makes the PRDC price = 100.")
print(f"\nFor reference: JPY deposit rate = {R_DOM*100:.1f}%")
print(f"Enhancement: {(par_coupon - R_DOM)*100:.2f}% above risk-free")

with apply_theme():
    fig, ax = create_figure(1)
    ax.plot(test_coupons * 100, prices_by_cpn, 'o-', linewidth=2)
    ax.axhline(NOTIONAL, color='black', linewidth=0.5, linestyle='--', label='Par (100)')
    ax.axvline(par_coupon * 100, color='tab:red', linewidth=1, linestyle=':', label=f'Par coupon ({par_coupon*100:.1f}%)')
    ax.set_xlabel('Fixed Coupon (%)')
    ax.set_ylabel('PRDC Price')
    ax.set_title('PRDC Price vs Fixed Coupon')
    ax.legend(fontsize=9)
    fig.tight_layout()

## Summary

| Component | What it does |
|-----------|-------------|
| **3-factor MC** | Domestic rate (OU) + foreign rate (OU) + FX (GBM with rate drift) |
| **Coupon** | `max(fixed + participation × (FX/strike - 1), 0)` — floored at zero |
| **Callable** | Issuer call via LSM — regression on (FX, rate) state |
| **Call value** | Non-callable price - callable price — what the investor gives up |
| **Correlation** | ρ(FX, domestic) is the key risk — drives natural hedging |

**Key insights:**
- The PRDC is essentially a **strip of FX call options** embedded in a bond
- The investor is **long FX vol** and **long FX spot** — benefits from JPY weakness
- The **issuer call** is valuable because it limits the issuer's exposure when FX moves against them
- **Correlation risk** is the hidden danger: ρ(FX, domestic rate) is hard to hedge and can swing the price significantly
- The **par coupon** is much higher than JPY deposits — that's the attraction for Japanese investors
- In practice, the structurer hedges with FX options + rate swaps + correlation swaps (if available)